# 01 · Bối cảnh và kiểm tra dữ liệu phân tích
Chạy SQL 00–08 trước, xử lý FATAL và xem kết quả đối soát cuối file 08. Profiling dữ liệu thô nằm ở SQL 03; notebook này kiểm tra grain và coverage của `mart.vw_sales`, không nạp hoặc làm sạch lại CSV.

Trong Jupyter/VS Code: chọn kernel đã cài requirements, **Restart Kernel → Run All**. Kết nối dùng Windows authentication; đặt `OLIST_SERVER` trước khi mở Jupyter nếu khác localhost. `.env` không tự nạp. Các tỷ lệ có đơn vị 0–1, tiền BRL.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'sql/07_create_mart_views.sql').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the project or notebooks folder.')
sys.path.insert(0, str(ROOT / 'notebooks'))
from utils.db import get_engine, read_mart
START, END = pd.Timestamp('2017-01-01'), pd.Timestamp('2018-09-01')
EXPORT_RESULTS = False  # Set True to save the displayed tables and figures.
OUT = ROOT / 'data/exports/sql_mart'
tables, figures = {}, {}
def cohort(frame):
    return frame.loc[frame.order_purchase_timestamp.ge(START)
                     & frame.order_purchase_timestamp.lt(END)].copy()

In [ ]:
engine = get_engine()
try:
    sales_all = read_mart('sales', engine)
finally:
    engine.dispose()
assert sales_all.order_id.is_unique, 'vw_sales must have one row per order.'
sales = cohort(sales_all)
assert not sales.empty, 'No orders in the selected period.'
assert sales.is_eligible.isin([0, 1]).all()
assert sales.loc[sales.is_eligible.eq(0), 'is_late'].isna().all()
coverage = pd.DataFrame({
    'metric': ['all_history_orders', 'cohort_orders', 'missing_merchandise', 'missing_payment', 'eligible_orders'],
    'orders': [len(sales_all), len(sales), sales.merchandise_value.isna().sum(),
               sales.total_payment.isna().sum(), sales.is_eligible.eq(1).sum()]})
tables['overview_coverage'] = coverage
display(coverage)

## Quy mô, trạng thái và tiền
Giá trị hàng/freight/AOV chính chỉ dùng delivered. Thanh toán báo riêng cho mọi status. AOV dùng tổng giá hàng chia số đơn delivered; số đơn thiếu item được công bố để không ngầm đổi mẫu số. Giá trị thiếu giữ NULL, không tự coi là 0.

In [ ]:
status = sales.groupby('order_status', dropna=False).size().rename('orders').reset_index()
delivered = sales[sales.order_status.eq('delivered')]
monthly = delivered.groupby('order_month').agg(
    delivered_orders=('order_id', 'size'), orders_with_items=('merchandise_value', 'count'),
    merchandise=('merchandise_value', lambda s: s.sum(min_count=1)),
    freight=('freight_value', lambda s: s.sum(min_count=1))).reset_index()
monthly['merchandise_aov'] = monthly.merchandise / monthly.delivered_orders
monthly['orders_without_items'] = monthly.delivered_orders - monthly.orders_with_items
payments = sales.groupby('order_month').agg(
    orders=('order_id', 'size'), payment_orders=('total_payment', 'count'),
    payment_value=('total_payment', lambda s: s.sum(min_count=1))).reset_index()
monthly['prior_month_orders'] = monthly.delivered_orders.shift(1)
tables.update(overview_status=status, overview_monthly=monthly, overview_payments=payments)
display(status); display(monthly); display(payments)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(monthly.order_month, monthly.delivered_orders)
axes[0].set(title='Delivered orders by purchase month', ylabel='Orders')
axes[1].plot(monthly.order_month, monthly.merchandise)
axes[1].set(title='Delivered merchandise value', ylabel='BRL')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()
figures['overview_monthly'] = fig

## Ghi nhận của người phân tích
Đọc các tháng thưa, thay đổi số đơn và coverage trước khi diễn giải. Khoảng thời gian chính là 2017-01 đến hết 2018-08; dữ liệu 2016 vẫn tồn tại trong mart để xác định đơn đầu toàn lịch sử. Không dùng giá trị thanh toán làm doanh thu nền tảng.

In [ ]:
if EXPORT_RESULTS:
    OUT.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(OUT / f'{name}.csv', index=False, encoding='utf-8-sig')
    for name, figure in figures.items():
        figure.savefig(OUT / f'{name}.png', dpi=160, bbox_inches='tight')
    print(f'Exported SQL mart results to {OUT}')
else:
    print('Results displayed only. Set EXPORT_RESULTS=True and Run All to export.')